# Constraint satisfaction

A **constraint satisfaction problem (CSP)** has **variables**, a **domain** of possible values for each, and **constraints** that limit which combinations are allowed. A solution gives every variable a value without breaking any constraint. Timetables, exam schedules, product configurations and puzzles all fit the mould.

The basic method is **backtracking**: give a value to one variable at a time, and as soon as a constraint is broken, undo the last choice and try the next value. Which variable to fill next matters a lot. The **minimum remaining values (MRV)** heuristic picks the variable with the fewest values still allowed: the most constrained one, where a wrong path shows up soonest.

## Colouring a map

The classic example: colour the states of Australia with three colours, so that no two neighbouring states share one. Tasmania (T) is an island with no neighbours.

In [1]:
states = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
colours = ["red", "green", "blue"]
borders = [("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"), ("SA", "Q"),
           ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V")]
neighbours = {state: set() for state in states}
for state_a, state_b in borders:
    neighbours[state_a].add(state_b)
    neighbours[state_b].add(state_a)

In [2]:
def allowed_colours(state, assignment):
    return [colour for colour in colours if all(assignment.get(neighbour) != colour for neighbour in neighbours[state])]


def colour_map(assignment, steps):
    if len(assignment) == len(states):
        return assignment
    unassigned = [state for state in states if state not in assignment]
    state = min(unassigned, key=lambda candidate: len(allowed_colours(candidate, assignment)))
    for colour in allowed_colours(state, assignment):
        steps.append(f"{state}={colour}")
        result = colour_map(assignment | {state: colour}, steps)
        if result:
            return result
    return None


steps = []
solution = colour_map({}, steps)
print(solution)
print(len(steps), "assignments:", " ".join(steps))

{'WA': 'red', 'NT': 'green', 'SA': 'blue', 'Q': 'red', 'NSW': 'green', 'V': 'red', 'T': 'red'}
7 assignments: WA=red NT=green SA=blue Q=red NSW=green V=red T=red


Seven assignments for seven states: with MRV the search never had to back up.

## Sudoku

A Sudoku is a CSP with 81 variables, the cells, each with the domain 1 to 9, and one kind of constraint: no digit twice in a row, a column or a 3×3 box. The solver below fills the empty cells by backtracking, and counts how many digits it places, wrong ones included. It can choose the next cell in two ways: the first empty cell in reading order, or the empty cell with the fewest digits still allowed (MRV).

In [3]:
puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]


def allowed_digits(grid, row, column):
    box_row, box_column = 3 * (row // 3), 3 * (column // 3)
    used = set(grid[row]) | {grid[r][column] for r in range(9)} | {
        grid[r][c] for r in range(box_row, box_row + 3) for c in range(box_column, box_column + 3)}
    return [digit for digit in range(1, 10) if digit not in used]

In [4]:
def solve_sudoku(grid, use_mrv, counter):
    empty_cells = [(row, column) for row in range(9) for column in range(9) if grid[row][column] == 0]
    if not empty_cells:
        return True
    if use_mrv:
        row, column = min(empty_cells, key=lambda cell: len(allowed_digits(grid, *cell)))
    else:
        row, column = empty_cells[0]
    for digit in allowed_digits(grid, row, column):
        grid[row][column] = digit
        counter[0] += 1
        if solve_sudoku(grid, use_mrv, counter):
            return True
    grid[row][column] = 0
    return False

In [5]:
for use_mrv in [False, True]:
    grid = [row[:] for row in puzzle]
    placements = [0]
    solve_sudoku(grid, use_mrv, placements)
    print(f"{'MRV' if use_mrv else 'Reading order':13}: {placements[0]:5} digits placed")

Reading order:  4208 digits placed
MRV          :    51 digits placed


The puzzle has 51 empty cells, so 51 placements means no backtracking at all. The solution:

In [6]:
for row in grid:
    print(" ".join(str(digit) for digit in row))

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


Real solvers add **constraint propagation**: after each placement they remove the digit from the domains of the cells it constrains, and give up on a branch as soon as some cell has no digit left (**forward checking**). Industrial problems are solved by dedicated tools that combine all these ideas: constraint and SAT solvers, such as Google's OR-Tools, for timetables and routes, and purpose-built resolvers in package managers (pip backtracks through versions; uv's resolver is based on the PubGrub algorithm).